# LeetCode 268 — Missing Number bằng Python

**Link đề:** https://leetcode.com/problems/missing-number/
**Độ khó:** Easy · **Chủ đề:** Array, Hash Table, Math, Binary Search, Bit Manipulation, Sorting

```python
class Solution:
    def missingNumber(self, nums: list[int]) -> int:
        ...
```

## Mục tiêu học
1. Đọc kỹ đề: `n` số **phân biệt** trong đoạn `[0, n]` → thiếu **đúng một** số.
2. Đi từ cách trực giác (sắp xếp, `set`) tới cách `O(1)` bộ nhớ đáp ứng follow-up.
3. Nắm **công thức Gauss** `0 + 1 + ... + n = n(n+1)/2` và mẹo "tổng kỳ vọng − tổng thực tế".
4. Hiểu **XOR** (`^`): `a ^ a = 0`, `a ^ 0 = a`, giao hoán — và dùng nó để "triệt tiêu cặp".
5. So sánh 4 lời giải, kèm biến thể (Find All Missing, thiếu 2 số, cyclic sort).

**Cách học:** đọc markdown → đoán output → chạy cell → so sánh. Cell TODO chỉ định nghĩa hàm, không tự chạy.

## 1. Đề bài

Cho mảng `nums` gồm `n` số **khác nhau**, mỗi số nằm trong đoạn `[0, n]`. Đoạn này có `n + 1` số, mảng có `n` số → **thiếu đúng một**. Trả về số bị thiếu.

| `nums` | `n` | Đoạn | Kết quả |
|---|---:|---|---:|
| `[3, 0, 1]` | 3 | `[0, 3]` | `2` |
| `[0, 1]` | 2 | `[0, 2]` | `2` (thiếu số cuối!) |
| `[9,6,4,2,3,5,7,0,1]` | 9 | `[0, 9]` | `8` |

**Constraints:** `n == len(nums)`, `1 <= n <= 10^4`, `0 <= nums[i] <= n`, các số **phân biệt**.

**Follow up:** Làm được với bộ nhớ phụ `O(1)` và thời gian `O(n)` không?

Chú ý ví dụ 2: số thiếu có thể là `n` (số lớn nhất) — một bẫy hay gặp.

In [1]:
nums = [3, 0, 1]
n = len(nums)
print("n =", n, "| đoạn đầy đủ:", list(range(n + 1)))
print("số không có trong nums:", [x for x in range(n + 1) if x not in nums])

n = 3 | đoạn đầy đủ: [0, 1, 2, 3]
số không có trong nums: [2]


## 2. Nền tảng Python

### 2.1. `range(n + 1)` và `len`

`range(a, b)` sinh `a, a+1, ..., b-1` (không gồm `b`). Muốn `0..n` **bao gồm** `n` thì viết `range(n + 1)`.

In [2]:
n = 3
print(list(range(n)), "<- thiếu n")
print(list(range(n + 1)), "<- đúng đoạn [0, n]")

[0, 1, 2] <- thiếu n
[0, 1, 2, 3] <- đúng đoạn [0, n]


### 2.2. `sorted`, `set` và phép `in`

- `sorted(nums)` → list mới đã sắp xếp, `O(n log n)`.
- `set(nums)` → tập hợp, `x in s` kiểm tra trung bình `O(1)` (so với `O(n)` của list).

In [3]:
nums = [9, 6, 4, 2, 3, 5, 7, 0, 1]
print(sorted(nums))
s = set(nums)
print("8 in s:", 8 in s, "| 7 in s:", 7 in s)

[0, 1, 2, 3, 4, 5, 6, 7, 9]
8 in s: False | 7 in s: True


### 2.3. `sum` và `enumerate`

- `sum(nums)` cộng mọi phần tử.
- `for i, x in enumerate(nums)` cho cả **chỉ số** `i` và **giá trị** `x`.

In [4]:
nums = [3, 0, 1]
print("sum =", sum(nums))
for i, x in enumerate(nums):
    print(f"i = {i}, x = {x}")

sum = 4
i = 0, x = 3
i = 1, x = 0
i = 2, x = 1


### 2.4. Phép XOR `^`

XOR so từng bit: giống nhau → 0, khác nhau → 1. Ba tính chất quan trọng:

| Tính chất | Ví dụ |
|---|---|
| `a ^ a = 0` | `5 ^ 5 = 0` |
| `a ^ 0 = a` | `5 ^ 0 = 5` |
| giao hoán, kết hợp | `a ^ b ^ a = b` |

Hệ quả: XOR một dãy mà mỗi số xuất hiện **hai lần**, trừ một số xuất hiện **một lần** → kết quả là số lẻ loi đó.

In [5]:
print("5 ^ 5 =", 5 ^ 5, "| 5 ^ 0 =", 5 ^ 0)
print(f"5 = {5:03b}, 3 = {3:03b}, 5 ^ 3 = {5 ^ 3:03b} = {5 ^ 3}")
print("3 ^ 7 ^ 3 =", 3 ^ 7 ^ 3)
acc = 0
for x in [4, 1, 2, 1, 2]:
    acc ^= x
print("XOR của [4,1,2,1,2] =", acc)

5 ^ 5 = 0 | 5 ^ 0 = 5
5 = 101, 3 = 011, 5 ^ 3 = 110 = 6
3 ^ 7 ^ 3 = 7
XOR của [4,1,2,1,2] = 4


## 3. Cách 1 — Sắp xếp rồi so vị trí

Sau khi sắp xếp, nếu không thiếu gì thì `sorted_nums[i] == i` với mọi `i`. Vị trí **đầu tiên** mà `sorted_nums[i] != i` chính là số thiếu. Nếu mọi vị trí đều khớp → thiếu `n`.

```text
[3,0,1] -> sắp xếp [0,1,3]
i=0: 0 == 0 ✔   i=1: 1 == 1 ✔   i=2: 3 != 2 ✘ -> thiếu 2
```

In [6]:
def missing_sort(nums: list[int]) -> int:
    s = sorted(nums)
    for i, x in enumerate(s):
        if x != i:
            return i
    return len(nums)          # mọi vị trí khớp -> thiếu số lớn nhất n

print(missing_sort([3, 0, 1]), missing_sort([0, 1]), missing_sort([9, 6, 4, 2, 3, 5, 7, 0, 1]))

2 2 8


**Chi phí:** `O(n log n)` thời gian, `O(n)` bộ nhớ (list sắp xếp). Nếu dùng `nums.sort()` tại chỗ thì đỡ bộ nhớ nhưng **làm hỏng input** của người gọi.

## 4. Cách 2 — `set`: hỏi từng số `0..n` "có mặt không?"

Đưa `nums` vào `set`, rồi duyệt `x = 0..n`; số đầu tiên không có trong set là đáp án. `O(n)` thời gian trung bình, nhưng `O(n)` bộ nhớ → **chưa** đạt follow-up.

In [7]:
def missing_set(nums: list[int]) -> int:
    present = set(nums)
    for x in range(len(nums) + 1):
        if x not in present:
            return x

print(missing_set([3, 0, 1]), missing_set([0, 1]), missing_set([9, 6, 4, 2, 3, 5, 7, 0, 1]))

2 2 8


**Lưu ý:** nếu thay `set` bằng list (`x not in nums`), mỗi lần kiểm tra là `O(n)` → tổng `O(n²)`. Đây là cái bẫy "trông giống `O(n)`".

## 5. Nút thắt và insight — "tổng kỳ vọng trừ tổng thực tế"

Muốn bộ nhớ `O(1)`, ta không được lưu "đã thấy số nào". Câu hỏi: **có đại lượng nào tóm tắt cả mảng chỉ bằng một con số** mà vẫn lộ ra số thiếu?

**Tổng!** Nếu không thiếu, tổng là `0 + 1 + ... + n`. Thiếu `m` thì tổng thực tế nhỏ hơn đúng `m`:

```text
missing = (0 + 1 + ... + n) - sum(nums)
```

**Công thức Gauss:** `0 + 1 + ... + n = n(n+1)/2`. Mẹo nhớ: ghép cặp `0+n, 1+(n-1), ...` mỗi cặp bằng `n`, có `(n+1)/2` cặp.

Ví dụ `[3, 0, 1]`: `n = 3`, kỳ vọng `3·4/2 = 6`, thực tế `4` → thiếu `2`.

In [8]:
for n in [3, 4, 10, 100]:
    print(f"n = {n}: sum(range(n+1)) = {sum(range(n + 1))}, n*(n+1)//2 = {n * (n + 1) // 2}")
nums = [3, 0, 1]
n = len(nums)
print("kỳ vọng =", n * (n + 1) // 2, "| thực tế =", sum(nums), "| thiếu =", n * (n + 1) // 2 - sum(nums))

n = 3: sum(range(n+1)) = 6, n*(n+1)//2 = 6
n = 4: sum(range(n+1)) = 10, n*(n+1)//2 = 10
n = 10: sum(range(n+1)) = 55, n*(n+1)//2 = 55
n = 100: sum(range(n+1)) = 5050, n*(n+1)//2 = 5050
kỳ vọng = 6 | thực tế = 4 | thiếu = 2


## 6. Cách 3 — Lời giải chính: công thức Gauss, giảng từng đoạn

**Đoạn 1:** `n = len(nums)` — **chú ý**: `n` là độ dài mảng, không phải `max(nums)` (vì có thể thiếu chính `n`).

**Đoạn 2:** `expected = n * (n + 1) // 2` — dùng `//` để giữ số nguyên (tích `n(n+1)` luôn chẵn nên chia hết).

**Đoạn 3:** `return expected - sum(nums)`.

Python có số nguyên không giới hạn nên không lo tràn số. (Ở Java/C++ với `n` lớn thì có thể tràn — lý do cách XOR được ưa chuộng ở đó.)

In [9]:
def missing_gauss(nums: list[int]) -> int:
    n = len(nums)
    expected = n * (n + 1) // 2
    return expected - sum(nums)

print(missing_gauss([3, 0, 1]), missing_gauss([0, 1]), missing_gauss([9, 6, 4, 2, 3, 5, 7, 0, 1]))

2 2 8


### 6.1. Phiên bản "cộng trừ xen kẽ" (tránh số trung gian lớn)

Thay vì tính tổng lớn rồi trừ, cộng dần `i - nums[i]` và bắt đầu từ `n`:

```text
missing = n + Σ (i - nums[i])
```

Giá trị trung gian luôn nhỏ — đây là cách viết an toàn tràn số ở ngôn ngữ khác.

In [10]:
def missing_running(nums: list[int]) -> int:
    missing = len(nums)
    for i, x in enumerate(nums):
        missing += i - x
    return missing

print(missing_running([3, 0, 1]), missing_running([0, 1]), missing_running([9, 6, 4, 2, 3, 5, 7, 0, 1]))

2 2 8


## 7. Cách 4 — XOR: triệt tiêu từng cặp

XOR tất cả **chỉ số** `0..n` với tất cả **giá trị** trong `nums`. Mỗi số có mặt xuất hiện **hai lần** (một lần là chỉ số, một lần là giá trị) → triệt tiêu thành 0. Số thiếu chỉ xuất hiện **một lần** (là chỉ số) → còn lại.

```text
nums = [3, 0, 1], n = 3
chỉ số:  0 ^ 1 ^ 2 ^ 3
giá trị: 3 ^ 0 ^ 1
tất cả:  (0^0) ^ (1^1) ^ (3^3) ^ 2 = 2
```

Khởi tạo `missing = n` (vì `enumerate` chỉ cho chỉ số `0..n-1`, phải tự thêm `n`).

In [11]:
def missing_xor(nums: list[int]) -> int:
    missing = len(nums)              # chỉ số n
    for i, x in enumerate(nums):
        missing ^= i ^ x
    return missing

print(missing_xor([3, 0, 1]), missing_xor([0, 1]), missing_xor([9, 6, 4, 2, 3, 5, 7, 0, 1]))

2 2 8


## 8. TRACE từng bước

### 8.1. Trace XOR với `nums = [3, 0, 1]`

| i | x | `i ^ x` | missing trước | missing sau |
|---:|---:|---:|---:|---:|
| — | — | — | — | 3 (khởi tạo = n) |
| 0 | 3 | 3 | 3 | 0 |
| 1 | 0 | 1 | 0 | 1 |
| 2 | 1 | 3 | 1 | 2 |

Kết quả `2`.

In [12]:
def trace_missing_xor(nums):
    missing = len(nums)
    print(f"nums = {nums}, khởi tạo missing = n = {missing} ({missing:04b})")
    for i, x in enumerate(nums):
        before = missing
        missing ^= i ^ x
        print(f"  i={i}, x={x}: i^x = {i ^ x:>2} | missing {before:>2} ({before:04b}) -> {missing:>2} ({missing:04b})")
    print("  => số thiếu:", missing)
    return missing

trace_missing_xor([3, 0, 1])
print()
trace_missing_xor([0, 1])

nums = [3, 0, 1], khởi tạo missing = n = 3 (0011)
  i=0, x=3: i^x =  3 | missing  3 (0011) ->  0 (0000)
  i=1, x=0: i^x =  1 | missing  0 (0000) ->  1 (0001)
  i=2, x=1: i^x =  3 | missing  1 (0001) ->  2 (0010)
  => số thiếu: 2

nums = [0, 1], khởi tạo missing = n = 2 (0010)
  i=0, x=0: i^x =  0 | missing  2 (0010) ->  2 (0010)
  i=1, x=1: i^x =  0 | missing  2 (0010) ->  2 (0010)
  => số thiếu: 2


2

### 8.2. Trace "cộng trừ xen kẽ" với `nums = [9,6,4,2,3,5,7,0,1]`

Cột `missing` sau mỗi bước = `n + Σ (i - nums[i])` tính tới đó.

In [13]:
def trace_missing_running(nums):
    missing = len(nums)
    print(f"khởi tạo missing = {missing}")
    for i, x in enumerate(nums):
        missing += i - x
        print(f"  i={i}, x={x}: cộng ({i} - {x}) = {i - x:>3} -> missing = {missing}")
    return missing

trace_missing_running([9, 6, 4, 2, 3, 5, 7, 0, 1])

khởi tạo missing = 9
  i=0, x=9: cộng (0 - 9) =  -9 -> missing = 0
  i=1, x=6: cộng (1 - 6) =  -5 -> missing = -5
  i=2, x=4: cộng (2 - 4) =  -2 -> missing = -7
  i=3, x=2: cộng (3 - 2) =   1 -> missing = -6
  i=4, x=3: cộng (4 - 3) =   1 -> missing = -5
  i=5, x=5: cộng (5 - 5) =   0 -> missing = -5
  i=6, x=7: cộng (6 - 7) =  -1 -> missing = -6
  i=7, x=0: cộng (7 - 0) =   7 -> missing = 1
  i=8, x=1: cộng (8 - 1) =   7 -> missing = 8


8

## 9. Code nộp LeetCode

In [14]:
class Solution:
    def missingNumber(self, nums: list[int]) -> int:
        n = len(nums)
        return n * (n + 1) // 2 - sum(nums)

class SolutionXor:
    def missingNumber(self, nums: list[int]) -> int:
        missing = len(nums)
        for i, x in enumerate(nums):
            missing ^= i ^ x
        return missing

for C in (Solution, SolutionXor):
    print(C.__name__, [C().missingNumber(a) for a in ([3, 0, 1], [0, 1], [9, 6, 4, 2, 3, 5, 7, 0, 1])])

Solution [2, 2, 8]
SolutionXor [2, 2, 8]


Khi nộp bản XOR, đổi tên `SolutionXor` thành `Solution`.

## 10. Vì sao đúng? + Độ phức tạp + so sánh

**Gauss:** Gọi `m` là số thiếu. Tập `nums` = `{0..n} \ {m}`, nên `sum(nums) = (0+...+n) − m` ⇒ `m = n(n+1)/2 − sum(nums)`.

**XOR — invariant:** sau khi xử lý chỉ số `0..i`, `missing` bằng XOR của `{n} ∪ {0..i}` với `{nums[0..i]}`. Khi hết vòng, mỗi số trong `{0..n}` khác `m` xuất hiện đúng hai lần (một lần chỉ số/khởi tạo, một lần giá trị) → triệt tiêu, chỉ còn `m`. Cần điều kiện **phân biệt** để mỗi số chỉ có mặt một lần trong `nums`.

**Cộng trừ xen kẽ:** chỉ là sắp xếp lại phép tính `Σ chỉ số − Σ giá trị`.

| Cách | Time | Space phụ | Ghi chú |
|---|---|---|---|
| 1. Sắp xếp | `O(n log n)` | `O(n)` (hoặc `O(1)` nếu sort tại chỗ, hỏng input) | trực giác |
| 2. Set | `O(n)` trung bình | `O(n)` | dễ hiểu |
| 3. Gauss | `O(n)` | `O(1)` | ngắn nhất, đạt follow-up |
| 4. XOR | `O(n)` | `O(1)` | không lo tràn số ở mọi ngôn ngữ |

## 11. Bộ test

In [15]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<30} PASS ({actual})")

sol = Solution()
check("mẫu 1", sol.missingNumber([3, 0, 1]), 2)
check("mẫu 2: thiếu n", sol.missingNumber([0, 1]), 2)
check("mẫu 3", sol.missingNumber([9, 6, 4, 2, 3, 5, 7, 0, 1]), 8)
check("n = 1, thiếu 1", sol.missingNumber([0]), 1)
check("n = 1, thiếu 0", sol.missingNumber([1]), 0)
check("thiếu 0", sol.missingNumber([2, 1, 3]), 0)
check("n = 10^4, thiếu 5000", sol.missingNumber([x for x in range(10001) if x != 5000]), 5000)

mẫu 1                          PASS (2)
mẫu 2: thiếu n                 PASS (2)
mẫu 3                          PASS (8)
n = 1, thiếu 1                 PASS (1)
n = 1, thiếu 0                 PASS (0)
thiếu 0                        PASS (0)
n = 10^4, thiếu 5000           PASS (5000)


In [16]:
import random
rng = random.Random(268)
funcs = [missing_sort, missing_set, missing_gauss, missing_running, missing_xor,
         Solution().missingNumber, SolutionXor().missingNumber]
for _ in range(2000):
    n = rng.randint(1, 60)
    m = rng.randint(0, n)
    nums = [x for x in range(n + 1) if x != m]
    rng.shuffle(nums)
    assert all(f(nums) == m for f in funcs), (nums, m)
print("4 cách (7 hàm) khớp nhau trên 2000 mảng ngẫu nhiên")

4 cách (7 hàm) khớp nhau trên 2000 mảng ngẫu nhiên


## 12. Lỗi người mới hay gặp

### Lỗi 1: Dùng `max(nums)` làm `n`

In [17]:
def wrong_max(nums):
    n = max(nums)                         # SAI khi số thiếu là n
    return n * (n + 1) // 2 - sum(nums)

print("wrong_max([0, 1]) =", wrong_max([0, 1]), "| đúng =", missing_gauss([0, 1]))

wrong_max([0, 1]) = 0 | đúng = 2


**Sửa:** `n = len(nums)` — đề cho độ dài bằng `n`.

### Lỗi 2: Sắp xếp xong quên trường hợp thiếu `n`

In [18]:
def wrong_sort(nums):
    s = sorted(nums)
    for i, x in enumerate(s):
        if x != i:
            return i
    # SAI: quên return len(nums) -> trả None

print("wrong_sort([0, 1]) =", wrong_sort([0, 1]), "| đúng = 2")

wrong_sort([0, 1]) = None | đúng = 2


### Lỗi 3: XOR quên khởi tạo bằng `n`

In [19]:
def wrong_xor(nums):
    missing = 0                           # SAI: thiếu chỉ số n
    for i, x in enumerate(nums):
        missing ^= i ^ x
    return missing

print("wrong_xor([3, 0, 1]) =", wrong_xor([3, 0, 1]), "| đúng = 2")

wrong_xor([3, 0, 1]) = 1 | đúng = 2


### Lỗi 4: Dùng `/` thay `//` → kết quả kiểu `float`

`3 * 4 / 2 - 4 = 2.0`. LeetCode có thể chấp nhận `2.0 == 2`, nhưng kiểu trả về sai (`float` thay vì `int`), và với `n` lớn `float` có thể mất chính xác.

In [20]:
n = 3
print(n * (n + 1) / 2 - 4, type(n * (n + 1) / 2 - 4).__name__)
print(n * (n + 1) // 2 - 4, type(n * (n + 1) // 2 - 4).__name__)

2.0 float
2 int


### Lỗi 5: `x not in nums` (list) trong vòng lặp → `O(n²)`

In [21]:
import time
big = list(range(10000))                  # thiếu 10000 -> phải quét đến cuối
t0 = time.perf_counter()
ans = next(x for x in range(len(big) + 1) if x not in big)      # list: O(n^2)
t1 = time.perf_counter()
ans2 = missing_set(big)
t2 = time.perf_counter()
print(f"list: {t1 - t0:.3f}s, set: {t2 - t1:.4f}s, cùng đáp án {ans} == {ans2}")

list: 0.273s, set: 0.0008s, cùng đáp án 10000 == 10000


## 13. Mẹo nhận dạng pattern

- **"Các số trong `[0, n]` (hoặc `[1, n]`), phân biệt / thiếu một / thừa một"** → nghĩ ngay tới **tổng Gauss** hoặc **XOR chỉ số với giá trị**.
- **"Mọi số xuất hiện 2 lần trừ một số"** → XOR (bài 136 Single Number).
- **"Giá trị nằm trong `[1, n]` và mảng dài `n`"** + yêu cầu `O(1)` bộ nhớ → **dùng chính mảng làm bảng băm** (cyclic sort, đánh dấu âm) — bài 448, 442, 41.
- Mảng **đã sắp xếp** → binary search: tìm `i` đầu tiên có `nums[i] != i`.

## 14. Biến thể

### 14.1. Mảng đã sắp xếp → binary search `O(log n)`

Bên trái số thiếu, `nums[i] == i`; từ số thiếu trở đi, `nums[i] == i + 1`. Tìm vị trí đầu tiên `nums[i] != i`.

In [22]:
def missing_sorted_bs(nums: list[int]) -> int:
    lo, hi = 0, len(nums)            # đáp án nằm trong [0, n]
    while lo < hi:
        mid = (lo + hi) // 2
        if nums[mid] == mid:         # chưa tới chỗ thiếu -> đi phải
            lo = mid + 1
        else:
            hi = mid
    return lo

print(missing_sorted_bs([0, 1, 3]), missing_sorted_bs([0, 1, 2]), missing_sorted_bs([1, 2, 3]))

2 3 0


### 14.2. Cyclic sort — đặt mỗi số về "đúng nhà"

Đổi chỗ để `nums[i] == i` khi có thể (bỏ qua giá trị `n` vì không có ô `n`). Sau đó tìm ô sai. `O(n)` thời gian, `O(1)` bộ nhớ, nhưng **thay đổi input**.

In [23]:
def missing_cyclic(nums: list[int]) -> int:
    a = nums[:]                       # copy để không phá dữ liệu mẫu
    i, n = 0, len(a)
    while i < n:
        v = a[i]
        if v < n and a[v] != v:
            a[i], a[v] = a[v], a[i]   # đưa v về ô v
        else:
            i += 1
    for i in range(n):
        if a[i] != i:
            return i
    return n

print(missing_cyclic([3, 0, 1]), missing_cyclic([0, 1]), missing_cyclic([9, 6, 4, 2, 3, 5, 7, 0, 1]))

2 2 8


### 14.3. Thiếu **hai** số trong `[1, n]`

Tổng cho `a + b`, tổng bình phương cho `a² + b²` → giải hệ. (Hoặc XOR + tách theo một bit khác nhau.)

In [24]:
import math
def two_missing(nums: list[int], n: int) -> tuple[int, int]:
    s = n * (n + 1) // 2 - sum(nums)                      # a + b
    q = n * (n + 1) * (2 * n + 1) // 6 - sum(x * x for x in nums)   # a^2 + b^2
    d = math.isqrt(2 * q - s * s)                          # (a - b)^2 = 2q - s^2
    a, b_ = (s - d) // 2, (s + d) // 2
    return a, b_

print(two_missing([1, 2, 4, 6], 6), two_missing([2, 3], 4))

(3, 5) (1, 4)


## 15. Bài tập (tự làm — cell chỉ định nghĩa, không gọi)

**Bài 1 — Tính tay:** `nums = [5, 2, 0, 4, 1]`. Tính `n`, tổng kỳ vọng, tổng thực tế và số thiếu.

In [25]:
# TODO: điền các con số
ex1_n = None
ex1_expected_sum = None
ex1_actual_sum = None
ex1_missing = None
# assert (ex1_n, ex1_expected_sum, ex1_actual_sum, ex1_missing) == (<xem đáp án>)

**Bài 2 — Điền chỗ trống: bản XOR.**

In [26]:
def exercise_missing_xor(nums: list[int]) -> int:
    # TODO 1: khởi tạo missing = ?
    # TODO 2: với mỗi (i, x), missing ^= ?
    pass

# assert exercise_missing_xor([3, 0, 1]) == 2 and exercise_missing_xor([0, 1]) == 2

**Bài 3 — Biến thể `[1, n]`:** mảng dài `n - 1` chứa các số phân biệt trong `[1, n]`, tìm số thiếu.

In [27]:
def exercise_missing_one_based(nums: list[int]) -> int:
    # TODO: chú ý n bây giờ là len(nums) + 1
    pass

# assert exercise_missing_one_based([1, 2, 4, 5]) == 3 and exercise_missing_one_based([2]) == 1

**Bài 4 — Find the Duplicate (dễ):** mảng dài `n + 1`, chứa `0..n-1` đủ cả và **thêm một** số lặp lại. Tìm số lặp bằng tổng hoặc XOR.

In [28]:
def exercise_extra_number(nums: list[int]) -> int:
    # TODO
    pass

# assert exercise_extra_number([0, 1, 2, 2]) == 2 and exercise_extra_number([1, 0, 0]) == 0

**Bài 5 — Find All Numbers Disappeared (LeetCode 448):** mảng dài `n`, giá trị trong `[1, n]`, có số lặp; trả về **tất cả** số trong `[1, n]` không xuất hiện. Thử làm với bộ nhớ phụ `O(1)` (không tính output) bằng mẹo **đánh dấu âm**.

In [29]:
def exercise_find_disappeared(nums: list[int]) -> list[int]:
    # TODO
    pass

# assert exercise_find_disappeared([4, 3, 2, 7, 8, 2, 3, 1]) == [5, 6]

**Bài 6 — Mini challenge:** Mảng đã sắp xếp, thiếu một số trong `[0, n]`. Viết bằng binary search và đếm số lần truy cập mảng khi `n = 10^6`.

In [30]:
def exercise_missing_sorted(nums: list[int]) -> tuple[int, int]:
    # TODO: trả về (số thiếu, số lần đọc nums[...])
    pass

# big = [x for x in range(10**6 + 1) if x != 777777]
# assert exercise_missing_sorted(big)[0] == 777777 and exercise_missing_sorted(big)[1] <= 21

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | `n = len(nums)` | kỳ vọng `n(n+1)/2` | trừ tổng thực tế |
| 2 | mỗi số có mặt xuất hiện 2 lần | chỉ số `n` không có trong `enumerate` | `missing = len(nums)`, `missing ^= i ^ x` |
| 3 | đoạn bây giờ là `1..n` | `n = len(nums) + 1` | `n(n+1)//2 - sum(nums)` |
| 4 | ngược với bài gốc | tổng thực tế − tổng `0..n-1` | `m = len(nums) - 1`; đáp án `sum(nums) - m(m-1)/2` |
| 5 | dùng dấu của `nums[v-1]` để đánh dấu "v có mặt" | `abs()` vì ô có thể đã bị đổi dấu | ô còn dương ⇒ `i + 1` thiếu |
| 6 | so `nums[mid]` với `mid` | `lo, hi = 0, n` | đếm mỗi lần đọc `nums[mid]` |

## 17. Đáp án tham khảo

In [31]:
ans1 = (5, 15, 12, 3)
nums1 = [5, 2, 0, 4, 1]
assert ans1 == (len(nums1), 5 * 6 // 2, sum(nums1), missing_gauss(nums1))
print("Bài 1:", ans1)

def answer_missing_xor(nums):
    missing = len(nums)
    for i, x in enumerate(nums):
        missing ^= i ^ x
    return missing

def answer_missing_one_based(nums):
    n = len(nums) + 1
    return n * (n + 1) // 2 - sum(nums)

def answer_extra_number(nums):
    m = len(nums) - 1               # các số 0..m-1 đủ cả, cộng 1 số thừa
    return sum(nums) - m * (m - 1) // 2

def answer_find_disappeared(nums):
    a = nums[:]                     # copy để không làm hỏng dữ liệu test
    for x in a:
        idx = abs(x) - 1
        if a[idx] > 0:
            a[idx] = -a[idx]        # đánh dấu "idx+1 có mặt"
    return [i + 1 for i, v in enumerate(a) if v > 0]

def answer_missing_sorted(nums):
    reads = 0
    lo, hi = 0, len(nums)
    while lo < hi:
        mid = (lo + hi) // 2
        reads += 1
        if nums[mid] == mid:
            lo = mid + 1
        else:
            hi = mid
    return lo, reads

assert answer_missing_xor([3, 0, 1]) == 2 and answer_missing_xor([0, 1]) == 2
assert answer_missing_one_based([1, 2, 4, 5]) == 3 and answer_missing_one_based([2]) == 1
assert answer_extra_number([0, 1, 2, 2]) == 2 and answer_extra_number([1, 0, 0]) == 0
assert answer_find_disappeared([4, 3, 2, 7, 8, 2, 3, 1]) == [5, 6]
big = [x for x in range(10**6 + 1) if x != 777777]
res = answer_missing_sorted(big)
assert res[0] == 777777 and res[1] <= 21
print("Bài 6: số thiếu", res[0], "sau", res[1], "lần đọc")
print("Tất cả đáp án PASS")

Bài 1: (5, 15, 12, 3)
Bài 6: số thiếu 777777 sau 20 lần đọc
Tất cả đáp án PASS


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def missingNumber(self, nums: list[int]) -> int:
        n = len(nums)
        return n * (n + 1) // 2 - sum(nums)      # hoặc XOR: missing = n; missing ^= i ^ x
```

**Câu thần chú:** *"Tổng kỳ vọng trừ tổng thực tế"* — *"XOR chỉ số với giá trị, cặp nào cũng tự hủy"*.

**Checklist:**
- [ ] `n = len(nums)`, **không** phải `max(nums)`.
- [ ] Đã xử lý trường hợp thiếu `0` và thiếu `n`.
- [ ] Dùng `//` để trả `int`.
- [ ] XOR: khởi tạo `missing = n`.
- [ ] Không dùng `x in list` trong vòng lặp (O(n²)).
- [ ] Tên method `missingNumber`, trả `int`.